In [1]:
import polars as pl, glob, os, re, time
from collections import defaultdict
print('polars', pl.__version__)

# Dò thư mục parquet_files/train thật
print('/kaggle/input:', os.listdir('/kaggle/input'))
hits = [h for h in glob.glob('/kaggle/input/**/train_applprev_2*.parquet', recursive=True)
        if '/parquet_files/train/' in h]
print('Tìm thấy:', hits[:3])
assert hits, 'Không thấy train_applprev_2*.parquet dưới /kaggle/input'
ROOT = os.path.dirname(hits[0])
print('ROOT =', ROOT)

TABLES = ['applprev_2', 'person_2', 'credit_bureau_b_2', 'credit_bureau_a_2']
N_BASE = 1_526_659
KEYS = ('case_id', 'num_group1', 'num_group2')

def part_idx(p):
    m = re.search(r'_(\d+)\.parquet$', p)
    return int(m.group(1)) if m else -1

def files_of(t):
    fs = glob.glob(f'{ROOT}/train_{t}.parquet') + glob.glob(f'{ROOT}/train_{t}_*.parquet')
    return sorted(fs, key=part_idx)

def collect(lf):
    try:
        return lf.collect(engine='streaming')
    except TypeError:
        return lf.collect(streaming=True)

FILES = {t: files_of(t) for t in TABLES}
assert all(FILES[t] for t in TABLES), {t: len(FILES[t]) for t in TABLES}
for t in TABLES:
    fs = FILES[t]
    schemas = [dict(pl.scan_parquet(f).collect_schema()) for f in fs]
    same = all(s == schemas[0] for s in schemas)
    size = sum(os.path.getsize(f) for f in fs) / 1e6
    print(f'\n== {t}: {len(fs)} file, {size:,.1f} MB, schema nhất quán: {same}')
    if not same:
        for f, s in zip(fs, schemas):
            diff = {c: (schemas[0].get(c), s.get(c)) for c in set(s) | set(schemas[0])
                    if schemas[0].get(c) != s.get(c)}
            if diff: print('   khác:', os.path.basename(f), diff)
    by_suf = defaultdict(list)
    for c, d in schemas[0].items():
        by_suf['key' if c in KEYS else c[-1]].append(f'{c}:{d}')
    for suf, cols in sorted(by_suf.items()):
        print(f'  [{suf}] {len(cols)}: ' + ', '.join(cols))

polars 1.35.2
/kaggle/input: ['competitions']
Tìm thấy: ['/kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/train/train_applprev_2.parquet']
ROOT = /kaggle/input/competitions/home-credit-credit-risk-model-stability/parquet_files/train

== applprev_2: 1 file, 28.5 MB, schema nhất quán: True
  [L] 2: conts_type_509L:String, credacc_cards_status_52L:String
  [M] 1: cacccardblochreas_147M:String
  [key] 3: case_id:Int64, num_group1:Int64, num_group2:Int64

== person_2: 1 file, 7.6 MB, schema nhất quán: True
  [D] 1: empls_employedfrom_796D:String
  [L] 1: addres_role_871L:String
  [M] 5: addres_district_368M:String, addres_zip_823M:String, conts_role_79M:String, empls_economicalst_849M:String, empls_employer_name_740M:String
  [T] 1: relatedpersons_role_762T:String
  [key] 3: case_id:Int64, num_group1:Int64, num_group2:Int64

== credit_bureau_b_2: 1 file, 2.1 MB, schema nhất quán: True
  [A] 1: pmts_pmtsoverdue_635A:Float64
  [D] 1: pmts_date_1107D:String
  [

In [2]:
def survey(t):
    t0 = time.time()
    lf = pl.scan_parquet(FILES[t])
    per_case = collect(lf.group_by('case_id').agg(
        n_rows=pl.len(),
        g1_max=pl.col('num_group1').max(),
        g2_max=pl.col('num_group2').max(),
        n_g1=pl.col('num_group1').n_unique(),
    ))
    nulls = collect(lf.select(pl.len().alias('__n'), pl.all().exclude(KEYS).null_count()))
    n_rows = nulls['__n'][0]
    n_case = per_case.height
    print(f'\n== {t}  ({time.time()-t0:.0f}s)')
    print(f'  rows {n_rows:,} | cases {n_case:,} ({n_case/N_BASE:.1%} base)')
    q = [0.5, 0.9, 0.99, 1.0]
    for c in ['n_rows', 'n_g1', 'g1_max', 'g2_max']:
        vals = [per_case[c].quantile(x) for x in q]
        print(f'  {c:7s} p50/p90/p99/max: ' + ' / '.join(f'{v:,.0f}' for v in vals))
    nr = {c: nulls[c][0] / n_rows for c in nulls.columns if c != '__n'}
    print('  null rate:', ', '.join(f'{c} {v:.1%}' for c, v in sorted(nr.items(), key=lambda x: x[1])))
    return per_case

PER_CASE = {}
for t in TABLES:
    PER_CASE[t] = survey(t)


== applprev_2  (1s)
  rows 14,075,487 | cases 1,221,522 (80.0% base)
  n_rows  p50/p90/p99/max: 8 / 26 / 48 / 79
  n_g1    p50/p90/p99/max: 4 / 12 / 20 / 20
  g1_max  p50/p90/p99/max: 3 / 11 / 19 / 19
  g2_max  p50/p90/p99/max: 2 / 3 / 3 / 11
  null rate: cacccardblochreas_147M 0.8%, conts_type_509L 17.0%, credacc_cards_status_52L 97.6%

== person_2  (0s)
  rows 1,643,410 | cases 1,435,105 (94.0% base)
  n_rows  p50/p90/p99/max: 1 / 2 / 4 / 34
  n_g1    p50/p90/p99/max: 1 / 1 / 2 / 4
  g1_max  p50/p90/p99/max: 0 / 0 / 1 / 4
  g2_max  p50/p90/p99/max: 0 / 0 / 1 / 31
  null rate: addres_district_368M 0.0%, addres_zip_823M 0.0%, conts_role_79M 0.0%, empls_economicalst_849M 0.0%, empls_employer_name_740M 0.0%, addres_role_871L 95.9%, relatedpersons_role_762T 98.3%, empls_employedfrom_796D 99.6%

== credit_bureau_b_2  (0s)
  rows 1,286,755 | cases 36,447 (2.4% base)
  n_rows  p50/p90/p99/max: 36 / 68 / 108 / 301
  n_g1    p50/p90/p99/max: 2 / 4 / 7 / 21
  g1_max  p50/p90/p99/max: 1 / 3 / 6

In [3]:
# Cell 3 — Quét leakage thời gian ở depth 2
base = pl.read_parquet(f'{ROOT}/train_base.parquet', columns=['case_id', 'date_decision'])
if base.schema['date_decision'] == pl.String:
    base = base.with_columns(pl.col('date_decision').str.to_date('%Y-%m-%d'))
B = base.lazy().select(
    'case_id', 'date_decision',
    dec_m=pl.col('date_decision').dt.year().cast(pl.Int64) * 12
          + pl.col('date_decision').dt.month().cast(pl.Int64),
)

def report(q, unit):
    """q: LazyFrame gồm case_id, x (chênh lệch so với date_decision)."""
    vc = collect(q.group_by('x').agg(n=pl.len()))
    n_fail = vc.filter(pl.col('x').is_null())['n'].sum()
    vc = vc.drop_nulls('x').sort('x')
    N = vc['n'].sum()
    cum = vc['n'].cum_sum() / N
    qt = lambda p: vc['x'][int((cum >= p).cast(pl.Int8).arg_max())]
    pos = vc.filter(pl.col('x') > 0)
    n_pos = pos['n'].sum()
    n_case_pos = collect(q.filter(pl.col('x') > 0).select(pl.col('case_id').n_unique())).item()
    print(f'  n {N:,} (parse lỗi {n_fail:,}) | min {vc["x"].min()} | p01 {qt(.01)} | p50 {qt(.5)} '
          f'| p99 {qt(.99)} | max {vc["x"].max()} {unit}')
    print(f'  ==0: {vc.filter(pl.col("x") == 0)["n"].sum()/N:.3%} | >0: {n_pos/N:.3%} '
          f'({n_pos:,} dòng, {n_case_pos:,} case)')
    if pos.height:
        print('  phân bố >0 (20 giá trị đầu):', dict(zip(pos['x'].to_list()[:20], pos['n'].to_list()[:20])))

def days_q(t, col):
    return (pl.scan_parquet(FILES[t]).select('case_id', col)
            .filter(pl.col(col).is_not_null())
            .join(B, on='case_id', how='left')
            .select('case_id', x=(pl.col(col).str.to_date('%Y-%m-%d', strict=False)
                                  - pl.col('date_decision')).dt.total_days()))

def month_q(ycol, mcol):
    return (pl.scan_parquet(FILES['credit_bureau_a_2']).select('case_id', ycol, mcol)
            .filter(pl.col(ycol).is_not_null() & pl.col(mcol).is_not_null())
            .join(B, on='case_id', how='left')
            .select('case_id', x=pl.col(ycol).cast(pl.Int64) * 12
                                 + pl.col(mcol).cast(pl.Int64) - pl.col('dec_m')))

t0 = time.time()

# (a) Cột D
for t, col in [('person_2', 'empls_employedfrom_796D'), ('credit_bureau_b_2', 'pmts_date_1107D')]:
    print(f'\n== {t}.{col} (ngày so với date_decision)')
    report(days_q(t, col), 'ngày')

# (b) credit_bureau_a_2: miền giá trị year/month
CB_A_PAIRS = [('pmts_year_1139T', 'pmts_month_158T'), ('pmts_year_507T', 'pmts_month_706T')]
for y, m in CB_A_PAIRS:
    r = collect(pl.scan_parquet(FILES['credit_bureau_a_2']).select(
        y_min=pl.col(y).min(), y_max=pl.col(y).max(),
        m_min=pl.col(m).min(), m_max=pl.col(m).max(),
        y_khong_nguyen=(pl.col(y) != pl.col(y).round()).sum(),
        lech_null=(pl.col(y).is_null() != pl.col(m).is_null()).sum(),
    ))
    print(f'\n== credit_bureau_a_2 {y}/{m}:', r.to_dicts()[0])

# (c) credit_bureau_a_2: chênh lệch theo tháng
for y, m in CB_A_PAIRS:
    print(f'\n== credit_bureau_a_2 ({y}, {m}) — tháng so với tháng của date_decision')
    report(month_q(y, m), 'tháng')

# (d) applprev_2 theo ROW_FILTER của applprev_1
ap1 = (pl.scan_parquet(files_of('applprev_1'))
       .select('case_id', 'num_group1', 'creationdate_885D')
       .join(B, on='case_id', how='left')
       .with_columns(x=(pl.col('creationdate_885D').str.to_date('%Y-%m-%d', strict=False)
                        - pl.col('date_decision')).dt.total_days()))
leak_keys = ap1.filter(pl.col('x') > 0).select('case_id', 'num_group1').unique()
ap2 = pl.scan_parquet(FILES['applprev_2']).select('case_id', 'num_group1')
K = ['case_id', 'num_group1']
print('\n== applprev_2 theo ROW_FILTER applprev_1 (creationdate_885D > 0)')
print('  cặp (case_id, num_group1) ở applprev_2 không có trong applprev_1:',
      f"{collect(ap2.unique().join(ap1.select(K), on=K, how='anti').select(pl.len())).item():,}")
print('  dòng applprev_2 bị loại:',
      f"{collect(ap2.join(leak_keys, on=K, how='semi').select(pl.len())).item():,}")
print('  case còn lại sau lọc:',
      f"{collect(ap2.join(leak_keys, on=K, how='anti').select(pl.col('case_id').n_unique())).item():,}")

print(f'\n({time.time()-t0:.0f}s)')


== person_2.empls_employedfrom_796D (ngày so với date_decision)
  n 5,757 (parse lỗi 0) | min -17125 | p01 -11344 | p50 -3069 | p99 -249 | max -34 ngày
  ==0: 0.000% | >0: 0.000% (0 dòng, 0 case)

== credit_bureau_b_2.pmts_date_1107D (ngày so với date_decision)
  n 1,286,755 (parse lỗi 0) | min -1118 | p01 -1085 | p50 -313 | p99 -2 | max 14 ngày
  ==0: 0.165% | >0: 0.646% (8,318 dòng, 5,159 case)
  phân bố >0 (20 giá trị đầu): {1: 1530, 2: 1482, 3: 1105, 4: 1322, 5: 822, 6: 742, 7: 473, 8: 310, 9: 199, 10: 135, 11: 120, 12: 53, 13: 22, 14: 3}

== credit_bureau_a_2 pmts_year_1139T/pmts_month_158T: {'y_min': 2008.0, 'y_max': 2021.0, 'm_min': 1.0, 'm_max': 12.0, 'y_khong_nguyen': 0, 'lech_null': 0}

== credit_bureau_a_2 pmts_year_507T/pmts_month_706T: {'y_min': 2000.0, 'y_max': 2028.0, 'm_min': 1.0, 'm_max': 12.0, 'y_khong_nguyen': 0, 'lech_null': 0}

== credit_bureau_a_2 (pmts_year_1139T, pmts_month_158T) — tháng so với tháng của date_decision
  n 66,129,816 (parse lỗi 0) | min -139 | p

In [4]:
# Cell 4 — DPD/overdue theo vị trí tháng so với tháng của date_decision (credit_bureau_a_2)
pl.Config(tbl_cols=-1, tbl_width_chars=220, float_precision=4)

FAMILIES = {
    'nhóm 1': ('pmts_year_1139T', 'pmts_month_158T', 'pmts_dpd_1073P', 'pmts_overdue_1140A'),
    'nhóm 2': ('pmts_year_507T',  'pmts_month_706T', 'pmts_dpd_303P',  'pmts_overdue_1152A'),
}

def bucket(x):
    return (pl.when(x < -12).then(pl.lit('a: < -12'))
            .when(x < 0).then(pl.lit('b: -12..-1'))
            .when(x == 0).then(pl.lit('c: 0'))
            .when(x <= 12).then(pl.lit('d: 1..12'))
            .otherwise(pl.lit('e: > 12')))

t0 = time.time()
CBA = pl.scan_parquet(FILES['credit_bureau_a_2'])

for name, (y, m, dpd, ovd) in FAMILIES.items():
    q = (CBA.select('case_id', y, m, dpd, ovd)
         .filter(pl.col(y).is_not_null())
         .join(B.select('case_id', 'dec_m'), on='case_id', how='left')
         .with_columns(x=pl.col(y).cast(pl.Int64) * 12 + pl.col(m).cast(pl.Int64) - pl.col('dec_m'))
         .group_by(bucket(pl.col('x')).alias('khoang'))
         .agg(n=pl.len(),
              dpd_null=pl.col(dpd).is_null().mean(),
              dpd_gt0=(pl.col(dpd) > 0).mean(),
              dpd_mean=pl.col(dpd).mean(),
              ovd_null=pl.col(ovd).is_null().mean(),
              ovd_gt0=(pl.col(ovd) > 0).mean(),
              ovd_mean=pl.col(ovd).mean()))
    print(f'\n== {name}: {dpd}, {ovd}')
    print(collect(q).sort('khoang'))

# Quan hệ giữa hai nhóm trên cùng dòng
y1, y2 = FAMILIES['nhóm 1'][0], FAMILIES['nhóm 2'][0]
d1, d2 = FAMILIES['nhóm 1'][2], FAMILIES['nhóm 2'][2]
print('\n== Chồng lấn giữa hai nhóm')
print(collect(CBA.select(
    ca_hai=(pl.col(y1).is_not_null() & pl.col(y2).is_not_null()).sum(),
    chi_nhom1=(pl.col(y1).is_not_null() & pl.col(y2).is_null()).sum(),
    chi_nhom2=(pl.col(y1).is_null() & pl.col(y2).is_not_null()).sum(),
    khong_nhom_nao=(pl.col(y1).is_null() & pl.col(y2).is_null()).sum(),
    dpd1_co_ma_year1_null=(pl.col(d1).is_not_null() & pl.col(y1).is_null()).sum(),
    dpd2_co_ma_year2_null=(pl.col(d2).is_not_null() & pl.col(y2).is_null()).sum(),
)).to_dicts()[0])

print(f'\n({time.time()-t0:.0f}s)')


== nhóm 1: pmts_dpd_1073P, pmts_overdue_1140A
shape: (4, 8)
┌────────────┬──────────┬──────────┬─────────┬──────────┬──────────┬─────────┬───────────┐
│ khoang     ┆ n        ┆ dpd_null ┆ dpd_gt0 ┆ dpd_mean ┆ ovd_null ┆ ovd_gt0 ┆ ovd_mean  │
│ ---        ┆ ---      ┆ ---      ┆ ---     ┆ ---      ┆ ---      ┆ ---     ┆ ---       │
│ str        ┆ u32      ┆ f64      ┆ f64     ┆ f64      ┆ f64      ┆ f64     ┆ f64       │
╞════════════╪══════════╪══════════╪═════════╪══════════╪══════════╪═════════╪═══════════╡
│ a: < -12   ┆ 19231072 ┆ 0.4987   ┆ 0.0608  ┆ 14.5472  ┆ 0.4982   ┆ 0.0616  ┆ 2390.5964 │
│ b: -12..-1 ┆ 28318395 ┆ 0.2424   ┆ 0.0577  ┆ 10.0773  ┆ 0.2387   ┆ 0.0584  ┆ 1804.6954 │
│ c: 0       ┆ 2655452  ┆ 0.0248   ┆ 0.0511  ┆ 7.3693   ┆ 0.0191   ┆ 0.0518  ┆ 1594.7251 │
│ d: 1..12   ┆ 15924897 ┆ 0.8980   ┆ 0.0343  ┆ 7.2056   ┆ 0.8973   ┆ 0.0346  ┆ 2177.3640 │
└────────────┴──────────┴──────────┴─────────┴──────────┴──────────┴─────────┴───────────┘

== nhóm 2: pmts_dpd_303P, pm

In [5]:
# Cell 5 — Top giá trị và số giá trị khác nhau của các cột chuỗi
STR_COLS = {
    'applprev_2': ['cacccardblochreas_147M', 'conts_type_509L', 'credacc_cards_status_52L'],
    'person_2':   ['addres_district_368M', 'addres_zip_823M', 'conts_role_79M',
                   'empls_economicalst_849M', 'empls_employer_name_740M',
                   'addres_role_871L', 'relatedpersons_role_762T'],
}

for t, cols in STR_COLS.items():
    lf = pl.scan_parquet(FILES[t])
    n = collect(lf.select(pl.len())).item()
    print(f'\n== {t} ({n:,} dòng)')
    for c in cols:
        vc = collect(lf.group_by(c).agg(k=pl.len())).sort('k', descending=True)
        top = ', '.join(f'{v}: {k/n:.1%}' for v, k in zip(vc[c].to_list()[:8], vc['k'].to_list()[:8]))
        print(f'  {c} | n_unique {vc.height:,} | {top}')

# person_2: phân bố theo num_group1 (0 = người nộp hồ sơ)
p2 = pl.scan_parquet(FILES['person_2'])
print('\n== person_2 theo num_group1')
print(collect(p2.group_by('num_group1').agg(
    dong=pl.len(), case=pl.col('case_id').n_unique()).sort('num_group1')))


== applprev_2 (14,075,487 dòng)
  cacccardblochreas_147M | n_unique 10 | a55475b1: 99.2%, None: 0.8%, P33_145_161: 0.0%, P201_63_60: 0.0%, P19_60_110: 0.0%, P23_105_103: 0.0%, P133_119_56: 0.0%, P41_107_150: 0.0%
  conts_type_509L | n_unique 10 | PRIMARY_MOBILE: 44.7%, HOME_PHONE: 17.6%, None: 17.0%, EMPLOYMENT_PHONE: 11.9%, PHONE: 6.6%, PRIMARY_EMAIL: 1.8%, ALTERNATIVE_PHONE: 0.3%, SECONDARY_MOBILE: 0.1%
  credacc_cards_status_52L | n_unique 7 | None: 97.6%, CANCELLED: 1.2%, ACTIVE: 0.8%, INACTIVE: 0.4%, BLOCKED: 0.0%, RENEWED: 0.0%, UNCONFIRMED: 0.0%

== person_2 (1,643,410 dòng)
  addres_district_368M | n_unique 508 | a55475b1: 96.3%, P125_48_164: 0.6%, P155_139_77: 0.2%, P114_74_190: 0.2%, P111_2_12: 0.2%, P215_163_136: 0.1%, P88_3_41: 0.1%, P37_84_33: 0.1%
  addres_zip_823M | n_unique 2,027 | a55475b1: 95.9%, P161_14_174: 0.4%, P144_138_111: 0.2%, P46_103_143: 0.2%, P85_138_173: 0.1%, P118_161_181: 0.1%, P11_15_81: 0.1%, P212_16_169: 0.1%
  conts_role_79M | n_unique 11 | a55475b1

In [6]:
# Cell 6 — Aggregate applprev_2, person_2, credit_bureau_b_2
PH = 'a55475b1'
TRAIN_MAX_WEEK = 43          # configs/config.yaml: train = WEEK_NUM 0–43
OUT = '/kaggle/working'

base_w = pl.read_parquet(f'{ROOT}/train_base.parquet', columns=['case_id', 'WEEK_NUM'])
TRAIN_IDS = base_w.filter(pl.col('WEEK_NUM') <= TRAIN_MAX_WEEK).select('case_id').lazy()

def null_ph(cols):
    return [pl.when(pl.col(c) == PH).then(None).otherwise(pl.col(c)).alias(c) for c in cols]

def vocab(lf, col, min_share=0.001):
    vc = collect(lf.join(TRAIN_IDS, on='case_id', how='semi')
                   .filter(pl.col(col).is_not_null())
                   .group_by(col).agg(k=pl.len()))
    vc = vc.filter(pl.col('k') / vc['k'].sum() >= min_share)
    return sorted(vc[col].to_list())

def counts(t, col, values):
    return [(pl.col(col) == v).sum().alias(f'{t}__{col}__n_{v}') for v in values]

def nuniq(name, col, cond=None):
    s = pl.col(col) if cond is None else pl.col(col).filter(cond)
    return s.drop_nulls().n_unique().alias(f'{name}__{col}__nunique')

def asum(c, name):
    return pl.when(pl.col(c).count() > 0).then(pl.col(c).sum()).alias(f'{name}__{c}__sum')

def days(col):
    return (pl.col(col).str.to_date('%Y-%m-%d', strict=False) - pl.col('date_decision')).dt.total_days()

def finish(t, df):
    assert df['case_id'].n_unique() == df.height, t
    df.write_parquet(f'{OUT}/d2_{t}.parquet')
    print(f'{t}: {df.shape[0]:,} case × {df.shape[1]-1} feature')
    return df

t0 = time.time()

# ---------- applprev_2: lan truyền ROW_FILTER từ applprev_1 ----------
t = 'applprev_2'
lf = (pl.scan_parquet(FILES[t])
      .join(leak_keys, on=K, how='anti')
      .with_columns(null_ph(['cacccardblochreas_147M'])))
aggs = [pl.len().alias(f'{t}__count'),
        pl.col('num_group1').n_unique().alias(f'{t}__n_prev')]
for c in ['conts_type_509L', 'credacc_cards_status_52L', 'cacccardblochreas_147M']:
    aggs += [nuniq(t, c)] + counts(t, c, vocab(lf, c))
d2_ap = finish(t, collect(lf.group_by('case_id').agg(aggs)))

# ---------- person_2 ----------
t = 'person_2'
M_P2 = ['addres_district_368M', 'addres_zip_823M', 'conts_role_79M',
        'empls_economicalst_849M', 'empls_employer_name_740M']
lf = (pl.scan_parquet(FILES[t])
      .with_columns(null_ph(M_P2))
      .join(B.select('case_id', 'date_decision'), on='case_id', how='left')
      .with_columns(emp=days('empls_employedfrom_796D'))
      .with_columns(emp=pl.when(pl.col('emp') > 0).then(None).otherwise(pl.col('emp'))))  # > 0 → null
APP = pl.col('num_group1') == 0
aggs = [pl.len().alias(f'{t}__count'),
        APP.sum().alias(f'{t}__app__count'),
        pl.col('num_group1').filter(~APP).n_unique().alias(f'{t}__n_related'),
        pl.col('emp').min().alias(f'{t}__empls_employedfrom_796D__min'),
        pl.col('emp').max().alias(f'{t}__empls_employedfrom_796D__max')]
for c in ['addres_district_368M', 'addres_zip_823M', 'empls_employer_name_740M']:
    aggs.append(nuniq(f'{t}__app', c, APP))
for c in ['conts_role_79M', 'empls_economicalst_849M', 'addres_role_871L', 'relatedpersons_role_762T']:
    aggs += [nuniq(t, c)] + counts(t, c, vocab(lf, c))
d2_p2 = finish(t, collect(lf.group_by('case_id').agg(aggs)))

# ---------- credit_bureau_b_2: ROW_FILTER pmts_date_1107D > 0 ----------
t = 'credit_bureau_b_2'
DPD, OVD = 'pmts_dpdvalue_108P', 'pmts_pmtsoverdue_635A'
lf = (pl.scan_parquet(FILES[t])
      .join(B.select('case_id', 'date_decision'), on='case_id', how='left')
      .with_columns(d=days('pmts_date_1107D'))
      .filter(pl.col('d').is_null() | (pl.col('d') <= 0)))
L12 = pl.col('d') >= -365
aggs = [pl.len().alias(f'{t}__count'),
        pl.col('num_group1').n_unique().alias(f'{t}__n_contracts'),
        pl.col('d').min().alias(f'{t}__pmts_date_1107D__min'),
        pl.col('d').max().alias(f'{t}__pmts_date_1107D__max'),
        pl.col(DPD).max().alias(f'{t}__{DPD}__max'),
        pl.col(DPD).mean().alias(f'{t}__{DPD}__mean'),
        (pl.col(DPD) > 0).sum().alias(f'{t}__{DPD}__n_pos'),
        pl.col(DPD).filter(L12).max().alias(f'{t}__{DPD}__max_l12m'),
        pl.col('d').filter(pl.col(DPD) > 0).max().alias(f'{t}__{DPD}__last_pos_days'),
        pl.col(OVD).max().alias(f'{t}__{OVD}__max'),
        pl.col(OVD).mean().alias(f'{t}__{OVD}__mean'),
        asum(OVD, t),
        pl.col(OVD).filter(L12).max().alias(f'{t}__{OVD}__max_l12m')]
d2_cbb = finish(t, collect(lf.group_by('case_id').agg(aggs)))

print(f'({time.time()-t0:.0f}s)')

applprev_2: 1,208,039 case × 25 feature
person_2: 1,435,105 case × 43 feature
credit_bureau_b_2: 36,419 case × 13 feature
(9s)


In [7]:
# Cell 7 — credit_bureau_a_2: MASK_MONTH theo nhóm, rồi aggregate
MONTH_MAX = -1   # giữ x ≤ -1 (loại tháng quyết định và các tháng sau); đổi thành 0 nếu giữ tháng quyết định
t = 'credit_bureau_a_2'
CBA = pl.scan_parquet(FILES[t])
DEC_M = B.select('case_id', 'dec_m')

def agg_family(y, m, dpd, ovd):
    lf = (CBA.select('case_id', 'num_group1', y, m, dpd, ovd)
          .filter(pl.col(y).is_not_null())
          .join(DEC_M, on='case_id', how='left')
          .with_columns(x=pl.col(y).cast(pl.Int64) * 12 + pl.col(m).cast(pl.Int64) - pl.col('dec_m'))
          .filter(pl.col('x') <= MONTH_MAX)                                   # MASK_MONTH
          .with_columns(dpd_l12=pl.when(pl.col('x') >= -12).then(pl.col(dpd)),
                        ovd_l12=pl.when(pl.col('x') >= -12).then(pl.col(ovd)),
                        pos_x=pl.when(pl.col(dpd) > 0).then(pl.col('x'))))
    aggs = [pl.len().alias(f'{t}__{y}__n_months'),
            pl.col('num_group1').n_unique().alias(f'{t}__{y}__n_contracts'),
            pl.col('x').min().alias(f'{t}__{y}__min_m'),
            pl.col(dpd).max().alias(f'{t}__{dpd}__max'),
            pl.col(dpd).mean().alias(f'{t}__{dpd}__mean'),
            (pl.col(dpd) > 0).sum().alias(f'{t}__{dpd}__n_pos'),
            pl.col('dpd_l12').max().alias(f'{t}__{dpd}__max_l12m'),
            pl.col('pos_x').max().alias(f'{t}__{dpd}__last_pos_m'),
            pl.col(ovd).max().alias(f'{t}__{ovd}__max'),
            pl.col(ovd).mean().alias(f'{t}__{ovd}__mean'),
            asum(ovd, t),
            pl.col('ovd_l12').max().alias(f'{t}__{ovd}__max_l12m')]
    t1 = time.time()
    df = collect(lf.group_by('case_id').agg(aggs))
    print(f'  {y}: {df.height:,} case, {df[f"{t}__{y}__n_months"].sum():,} tháng hợp lệ ({time.time()-t1:.0f}s)')
    return df

fams = [agg_family(*FAMILIES[k]) for k in FAMILIES]
d2_cba = fams[0].join(fams[1], on='case_id', how='full', coalesce=True)
d2_cba = finish(t, d2_cba)

  pmts_year_1139T: 1,262,123 case, 47,549,467 tháng hợp lệ (7s)
  pmts_year_507T: 1,014,002 case, 154,443,863 tháng hợp lệ (19s)
credit_bureau_a_2: 1,384,357 case × 24 feature


In [8]:
# Cell 8 — Ghép depth2_agg.parquet
from collections import Counter

ids = pl.read_parquet(f'{ROOT}/train_base.parquet', columns=['case_id'])
parts = {'applprev_2': d2_ap, 'person_2': d2_p2,
         'credit_bureau_b_2': d2_cbb, 'credit_bureau_a_2': d2_cba}

d2 = ids
for t, df in parts.items():
    d2 = d2.join(df, on='case_id', how='left')

assert d2.height == N_BASE and d2['case_id'].n_unique() == N_BASE
assert len(set(d2.columns)) == len(d2.columns)
print(f'depth2_agg: {d2.height:,} × {d2.width} ({d2.width-1} feature)')
for t, df in parts.items():
    print(f'  {t}: {df.width-1} feature, phủ {df.height/N_BASE:.1%} case')
print('  kiểu dữ liệu:', dict(Counter(str(d) for d in d2.dtypes[1:])))

nr = d2.select(pl.all().exclude('case_id').null_count() / d2.height).row(0, named=True)
print('  cột toàn null:', [c for c, v in nr.items() if v == 1.0])
print('  cột hằng số (≤ 1 giá trị khác null):',
      [c for c in d2.columns[1:] if d2[c].drop_nulls().n_unique() <= 1])

path = f'{OUT}/depth2_agg.parquet'
d2.write_parquet(path)
print(f'  {path}: {os.path.getsize(path)/1e6:.2f} MB')

for t in parts:
    os.remove(f'{OUT}/d2_{t}.parquet')
print('  output:', os.listdir(OUT))

depth2_agg: 1,526,659 × 106 (105 feature)
  applprev_2: 25 feature, phủ 79.1% case
  person_2: 43 feature, phủ 94.0% case
  credit_bureau_b_2: 13 feature, phủ 2.4% case
  credit_bureau_a_2: 24 feature, phủ 90.7% case
  kiểu dữ liệu: {'UInt32': 75, 'Int64': 9, 'Float64': 21}
  cột toàn null: []
  cột hằng số (≤ 1 giá trị khác null): []
  /kaggle/working/depth2_agg.parquet: 57.55 MB
  output: ['depth2_agg.parquet', '__notebook__.ipynb']
